# Omega -> таблица «Omega Июль 2026»


Самый простой из трёх: АТС здесь нет, только реестр CRM.
Данные уходят на лист с датой как есть, с формулой округления
минут в колонке G и двумя пустыми колонками под ручное заполнение.

Порядок работы в DataSpell:
    1. Ячейка 1 — настройки
    2. Ячейка 2 — чтение и проверка на экране
    3. Ячейка 3 — выгрузка


## ЯЧЕЙКА 1 — НАСТРОЙКИ

In [ ]:
import re

import numpy as np
import pandas as pd

from etl_core import (
    DATA_DIR, CREDENTIALS, env,
    DateResolver, ExcelReader, MultiDay, SheetsLoader, Serializer,
    Palette, preview, log, month_title,
)

BASE_DIR = str(DATA_DIR)   # выгрузки лежат в папке data/ в корне репозитория
# CREDENTIALS импортируется из etl_core: credentials.json в корне, в git не попадает
# etl_core обновляется вместе с ноутбуками, а Python держит уже
# загруженный модуль в памяти: заменить файл мало, нужен Restart Kernel.
# Проверяем сразу, иначе это выяснится на середине выгрузки.
_need = ["open_sheet_for_date", "req_borders", "req_header"]
_missing = [m for m in _need if not hasattr(SheetsLoader, m)]
if _missing:
    raise ImportError(
        "etl_core.py устарел — в нём нет методов: " + ", ".join(_missing)
        + "\n\n1. Положите свежий etl_core.py в папку etl/"
        + "\n2. Нажмите Restart Kernel — без этого Python продолжит "
          "использовать старую версию из памяти"
    )

# Таблица заводится помесячно — ищется по названию,
# месяц подставляется из даты файла (см. комментарий в alpha_registry.ipynb).
SPREADSHEET_TITLE = "Omega {month} {year}"

USE_TITLE = True
SPREADSHEET_ID = env("OMEGA_SHEET_ID")

# Проект закрыт, в crm_reports.py его уже нет: файл качался вручную.
CRM_FILE = "omega_crm.xlsx"

HEADERS = [
    "ID звонка", "Направление звонка", "Номер телефона",
    "Время начала звонка", "Время конца звонка", "Продолжительность, сек",
    "Округление, минуты", "Удержание, сек", "Обработка, сек",
    "Оператор", "Тема звонка / вопросы", "Заявки", "Результат обзвона",
]

# None — колонка не из CRM: либо формула, либо пустая под ручной ввод
SOURCE = ["ID звонка", "Направление звонка", "Номер телефона",
          "Время начала звонка", "Время конца звонка",
          "Продолжительность, сек", None, "Удержание, сек",
          "Обработка, сек", "Оператор", "Тема звонка / вопросы", "", ""]

# Выгрузка CRM теряет первую запись периода. Поэтому файл берётся
# с запасом в один день назад: самый ранний день считается буферным,
# показывается на экране, но не грузится — именно у него откушена
# первая запись. Поставьте False, если буфер не нужен.
USE_BUFFER_DAY = True

# --- лист «Итог» ---
# После заливки дневного листа скрипт дописывает на «Итог» три числа
# за этот день: входящие, исходящие и сумму округлённых минут.
# Колонка «Все вызовы» там формула =СУММ(вх;исх) — её не трогаем,
# посчитается сама. Строка ищется по дате в колонке A, а не по
# номеру дня: если шапку листа сдвинут, слепой расчёт молча
# записал бы цифры не в ту дату.
UPDATE_TOTAL = True          # False — не трогать лист «Итог»
TOTAL_SHEET = "Итог"
TOTAL_FIRST_COL = "C"        # C входящие, D исходящие, E минуты

reader = ExcelReader(BASE_DIR)

## ЯЧЕЙКА 2 — ЧТЕНИЕ И РАЗБОР ПО ДНЯМ

In [ ]:
crm = reader.read_crm(CRM_FILE)
parts = MultiDay.split(crm, "Время начала звонка")
plan, DAYS_TO_LOAD, BUFFER_DAY = MultiDay.plan(parts, USE_BUFFER_DAY)

preview(plan, f"ЧТО В ФАЙЛЕ {CRM_FILE}")

if BUFFER_DAY is not None:
    b = parts[BUFFER_DAY]
    print(f"\nБуферный день {BUFFER_DAY.strftime('%d.%m.%Y')} — "
          f"{len(b)} строк, грузиться НЕ будет.")
    print("Вот его первые строки, посмотрите перед тем как отбросить:")
    preview(
        b.sort_values("Время начала звонка").head(5)[
            ["ID звонка", "Направление звонка", "Время начала звонка",
             "Продолжительность, сек", "Оператор"]],
        f"БУФЕРНЫЙ ДЕНЬ {BUFFER_DAY.strftime('%d.%m.%Y')} — НЕ ГРУЗИТСЯ"
    )

print(f"\nБудет загружено дней: {len(DAYS_TO_LOAD)}")
for d in DAYS_TO_LOAD:
    g = parts[d]
    v = g["Направление звонка"].value_counts().to_dict()
    print(f"   {DateResolver.sheet_name(d)}: {len(g):>4} строк, {v}")

if not DAYS_TO_LOAD:
    raise ValueError("Грузить нечего: в файле только буферный день. "
                     "Возьмите период шире.")

## ЯЧЕЙКА 3 — ВЫГРУЗКА

In [ ]:
def build_rows(day_df: pd.DataFrame) -> list:
    """Собирает строки листа: данные CRM, формула округления, пустые колонки."""
    rows = []
    for i, (_, row) in enumerate(day_df.iterrows()):
        r = i + 2                       # строка в Sheets
        line = []
        for c in SOURCE:
            if c is None:
                line.append(f"=CEILING(F{r}/60)")
            elif c == "":
                line.append("")
            else:
                line.append(Serializer.cast(row.get(c)))
        rows.append(line)
    return rows


def day_totals(day_df: pd.DataFrame) -> list:
    """
    Считает по дню три числа для листа «Итог».

    Минуты — сумма округлений ВВЕРХ, то есть ровно то, что даёт
    формула =CEILING(F/60) в колонке G дневного листа. Проверено
    на августе: цифры совпадают с теми, что стояли на «Итоге»
    до этого, день в день.
    """
    direction = day_df["Направление звонка"].astype(str).str.strip()
    sec = pd.to_numeric(day_df["Продолжительность, сек"],
                        errors="coerce").fillna(0)
    return [
        int((direction == "Входящий").sum()),
        int((direction == "Исходящий").sum()),
        int(np.ceil(sec / 60).sum()),
    ]


def update_total_sheet(loader, totals: list) -> list:
    """
    Дописывает посчитанное на лист «Итог».

    Строку под дату ищем в колонке A: там лежит текст вида
    «01.08.2026 c 00:00-23:59». Если строки за эту дату нет —
    пишем предупреждение и пропускаем, но не создаём новую:
    лист свёрстан вручную, дописывать в него вслепую опасно.
    """
    q = loader.quote_sheet
    try:
        grid = loader.batch_get([f"{q(TOTAL_SHEET)}!A1:A60"])[0]
    except Exception as e:
        log.warning("Лист «%s» прочитать не удалось (%s) — пропускаю",
                    TOTAL_SHEET, e)
        return []

    row_by_date = {}
    for i, r in enumerate(grid or [], start=1):
        m = re.match(r"\s*(\d{2})\.(\d{2})\.(\d{4})",
                     str(r[0]) if r else "")
        if m:
            row_by_date[(int(m[3]), int(m[2]), int(m[1]))] = i

    payload, done = [], []
    for d, vals in totals:
        row = row_by_date.get((d.year, d.month, d.day))
        if row is None:
            log.warning("На листе «%s» нет строки за %s",
                        TOTAL_SHEET, d.strftime("%d.%m.%Y"))
            continue
        payload.append({
            "range": f"{q(TOTAL_SHEET)}!{TOTAL_FIRST_COL}{row}",
            "values": [vals],
        })
        done.append((d, row, vals))

    if payload:
        # Пишем через таблицу, а не через loader.write: тот отправляет
        # в текущий открытый лист, а нам нужен другой.
        loader.sh.values_batch_update({
            "valueInputOption": "USER_ENTERED",
            "data": payload,
        })
        log.info("Лист «%s»: обновлено строк %d", TOTAL_SHEET, len(payload))
    return done


def upload_all() -> None:
    """
    Заливает каждый день на свой лист.

    Подключение к таблице одно на все дни: месяц у них общий,
    а переподключение на каждый день тратило бы квоту API впустую.
    """
    first = DAYS_TO_LOAD[0]
    loader = SheetsLoader(CREDENTIALS, SPREADSHEET_ID)
    if USE_TITLE:
        loader.connect(title=month_title(SPREADSHEET_TITLE, first))
    else:
        loader.connect()

    done, totals = [], []
    for d in DAYS_TO_LOAD:
        day_df = parts[d].sort_values("Время начала звонка")

        # Дни из разных месяцев на один лист не положить: таблица месячная
        if d.month != first.month or d.year != first.year:
            log.warning("%s из другого месяца — пропускаю. Загрузите его "
                        "отдельным запуском", DateResolver.sheet_name(d))
            continue

        loader.open_sheet_for_date(d, rows=max(1000, len(day_df) + 50), cols=20)
        rows = build_rows(day_df)
        loader.write([{"range": "A1", "values": [HEADERS] + rows}])
        loader.apply_format([
            loader.req_header(0, 0, len(HEADERS), Palette.HEADER_GREEN),
            loader.req_freeze(1),
            loader.req_borders(0, len(rows) + 1, 0, len(HEADERS)),
            loader.req_col_width(0, 1, 90),
            loader.req_col_width(3, 5, 150),
            loader.req_col_width(10, 11, 200),
        ], strict=False)
        done.append((DateResolver.sheet_name(d), len(rows)))
        totals.append((d, day_totals(day_df)))
        log.info("Лист %s: %d строк", DateResolver.sheet_name(d), len(rows))

    total_done = update_total_sheet(loader, totals) if UPDATE_TOTAL else []

    print(f"\nГотово: «{loader.sh.title}»")
    for name, n in done:
        print(f"   {name}: {n} строк")

    if total_done:
        print(f"\nЛист «{TOTAL_SHEET}» — записано:")
        print(f"   {'дата':<12}{'строка':>7}{'вход':>7}{'исход':>7}{'мин':>7}")
        for d, row, (vx, ix, mn) in total_done:
            print(f"   {d.strftime('%d.%m.%Y'):<12}{row:>7}"
                  f"{vx:>7}{ix:>7}{mn:>7}")
    if BUFFER_DAY is not None:
        print(f"\nБуферный день {BUFFER_DAY.strftime('%d.%m.%Y')} не загружался.")


upload_all()